In [59]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualQuantizer(nn.Module):
    def __init__(self, cb_dim, cb_size, beta):
        super().__init__()
        self.cb_dim = cb_dim
        self.cb_size = cb_size
        self.beta = beta

        self.codebook = nn.Embedding(cb_size, cb_dim) #(M, D)
        nn.init.uniform_(self.codebook.weight, -1.0/cb_size, 1.0/cb_size)

    def forward(self, z):
        z2 = torch.sum(z**2, dim=1, keepdim=True) # (B, 1)  / z: (B, D)
        cb2 = torch.sum(self.codebook.weight**2, dim=1).unsqueeze(0) # (1, M)
        cross = z @ self.codebook.weight.t()

        dist = z2 + cb2 - 2 * cross
        indices = torch.argmin(dist, dim=1)
        z_q = self.codebook(indices)

        z_q_st = z + (z_q - z).detach() 

        codebook_loss = F.mse_loss(z_q, z.detach())
        commitment_loss = F.mse_loss(z, z_q.detach())
        vq_loss = codebook_loss + self.beta * commitment_loss

        return z_q_st, indices, vq_loss
    

class RQVAE(nn.Module):
    def __init__(self, cb_dim, cb_size, beta, cb_level, input_dim):
        super().__init__()
        self.cb_dim = cb_dim
        self.cb_size = cb_size
        self.beta = beta
        self.cb_level = cb_level
        self.input_dim = input_dim
        hidden_dim = 2 * input_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, cb_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(cb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

        self.quantizers = nn.ModuleList(
            [ResidualQuantizer(cb_dim=cb_dim, cb_size=cb_size, beta=beta) for _ in range(cb_level)]
        )

    def quantize(self, z):
        z_q_all = torch.zeros_like(z, device=z.device)
        indices = []
        vq_loss = torch.tensor(0.0, device=z.device)

        residue = z.clone()
        for q in self.quantizers:
            z_q, idx, loss = q(residue)
            
            z_q_all = z_q_all + z_q
            indices.append(idx)
            vq_loss = vq_loss + loss

            residue = residue - z_q

        return z_q_all, indices, vq_loss

    def forward(self, x):
        z = self.encoder(x)
        z_q, indices, vq_loss = self.quantize(z)

        x_recon = self.decoder(z_q)    
        recon_loss = F.mse_loss(x_recon, x)

        return z_q, indices, recon_loss, vq_loss

    @torch.no_grad()
    def encode(self, x):
        z = self.encode(x)
        _, indices, _ = self.quantize(z)

        return indices, z 



In [60]:
input_dim = 6
cb_dim = 4
cb_size = 8
cb_level = 3
beta = 0.25

tokenizer = RQVAE(cb_size=cb_size, cb_level=cb_level, cb_dim=cb_dim, input_dim=emb_dim, beta=beta)

batch_size = 2

x = torch.randn(batch_size, input_dim)
x_recon, idx, recon_loss, vq_loss = tokenizer.forward(x)

print(x_recon)
print(idx)
print(recon_loss)
print(vq_loss)

tensor([[ 0.0486, -0.0088, -0.0716,  0.0679],
        [ 0.1819,  0.0748, -0.1337,  0.0352]], grad_fn=<AddBackward0>)
[tensor([4, 3]), tensor([7, 1]), tensor([3, 1])]
tensor(0.9350, grad_fn=<MseLossBackward0>)
tensor(0.0885, grad_fn=<AddBackward0>)


In [61]:
from torch.utils.data import TensorDataset, DataLoader

n_samples = 1024
input_dim = 8
n_clusters = 8

torch.manual_seed(42)

centers = torch.randn(n_clusters, input_dim) * 2.0
cluster_ids = torch.randint(0, n_clusters, (n_samples,))
X = centers[cluster_ids] + torch.randn(n_samples, input_dim) * 0.25 

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [63]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = RQVAE(input_dim=input_dim, cb_dim=4, cb_level=2, cb_size=8, beta=0.25).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCH = 200

for epoch in range(EPOCH):
    model.train()
    total_recon = total_vq = 0.0

    for batch in dataloader:
        x = batch[0].to(device)

        z_q, indices, recon_loss, vq_loss = model(x)

        loss = recon_loss + vq_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_recon = total_recon + recon_loss.item()
        total_vq = total_vq + vq_loss.item()

    if epoch % 20 ==0:
        print(f"Epoch {epoch:3d} | Recon loss: {total_recon / len(dataloader):.3f} | VQ loss: {total_vq / len(dataloader):.3f}")


device: cuda
Epoch   0 | Recon loss: 4.620 | VQ loss: 0.416
Epoch  20 | Recon loss: 2.069 | VQ loss: 15.442
Epoch  40 | Recon loss: 0.642 | VQ loss: 1.120
Epoch  60 | Recon loss: 0.334 | VQ loss: 0.396
Epoch  80 | Recon loss: 0.169 | VQ loss: 0.562
Epoch 100 | Recon loss: 0.089 | VQ loss: 0.285
Epoch 120 | Recon loss: 0.070 | VQ loss: 0.197
Epoch 140 | Recon loss: 0.070 | VQ loss: 0.164
Epoch 160 | Recon loss: 0.070 | VQ loss: 0.155
Epoch 180 | Recon loss: 0.073 | VQ loss: 0.148


In [ ]:
@torch.no_grad()
def check_code_usage(model, dataloader):
    model.eval()
    usage = [set() for _ in range(model.cb_level)]
    for batch in dataloader:
        x = batch[0].to(device)
        indices, _ = model.encode(x)
        print(indices.shape)
        #for lvl, idx in enumerate(indices)

check_code_usage(model, dataloader)